In [3]:
from databricks.connect import DatabricksSession
import os
from dotenv import load_dotenv
import gc
import marimo as mo
from datetime import datetime, timedelta
import pyspark.sql.functions as F
from pyspark.sql.functions import col
import polars as pl

load_dotenv()
gc.collect()

0

In [1]:

import os; os.getenv("DATABRICKS_HOST")

'dbc-7cf9cb2d-15bd.cloud.databricks.com'

In [2]:
import pyarrow.dataset as ds
import pyarrow.fs as pafs

In [3]:
spark = DatabricksSession.builder.remote(
            host=os.getenv("DATABRICKS_HOST"),
            token=os.getenv("DATABRICKS_TOKEN")
        ).serverless(True).getOrCreate()

# Configs
Block_Table_UC = "workspace.btc_data.blocks"
block_source_External_UC = "workspace.btc_data.blocks_source"
tx_source_External_UC = "workspace.btc_data.tx_source"
# paths
tx = 's3://aws-public-blockchain/v1.0/btc/transactions/'
#dates
current_date = spark.sql("select current_date()").collect()[0][0]

In [5]:
# auto loader test for the data on public s3, aws btc on chain data
df = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .load("s3://aws-public-blockchain/v1.0/btc/blocks/date=2026-09-01/")


In [9]:
# creating volume

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.btc_data.test_chk;")

DataFrame[]

In [11]:
df = (spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "parquet")
  .option("cloudFiles.schemaLocation",
          "/Volumes/workspace/btc_data/test_chk/schema")
  .load("s3://aws-public-blockchain/v1.0/btc/blocks/date=2026-09-01/")
)

In [13]:
(df.writeStream
  .option("checkpointLocation", "/Volumes/workspace/btc_data/test_chk/chk")
  .trigger(availableNow=True)
  .toTable("workspace.btc_data.test_blocks_autoload")
  .awaitTermination())

In [16]:
spark.sql("SELECT * FROM workspace.btc_data.test_blocks_autoload  ").show()

+--------------------+---------+-------------------+----------+--------+--------------------+--------------------+--------------------+-------+-------+--------------------+------+-----------------+--------------------+-------------+-------------------+----------+--------------------+-------------+
|                hash|  version|         mediantime|     nonce|    bits|          difficulty|           chainwork|   previousblockhash|   size| weight|      coinbase_param|number|transaction_count|         merkle_root|stripped_size|          timestamp|      date|       last_modified|_rescued_data|
+--------------------+---------+-------------------+----------+--------+--------------------+--------------------+--------------------+-------+-------+--------------------+------+-----------------+--------------------+-------------+-------------------+----------+--------------------+-------------+
|00000000000000000...|537198592|2026-09-01 19:43:48|3325809740|17023cc1|1.258070765471975E14|0000000000

In [3]:
# 1. Compare the physical plans — look at the "PartitionFilters" line
spark.sql("""EXPLAIN SELECT max(number) FROM workspace.btc_data.blocks_source
             WHERE date >= date_format(current_date - 1, 'yyyy-MM-dd')""").show(truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:

spark.sql("""EXPLAIN SELECT max(block_number) FROM workspace.btc_data.tx_source
             WHERE date >= date_format(current_date - 1, 'yyyy-MM-dd')""").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
rel = spark.table(tx_source_External_UC)

In [19]:
rel.printSchema()

root
 |-- txid: string (nullable = true)
 |-- hash: string (nullable = true)
 |-- version: long (nullable = true)
 |-- size: long (nullable = true)
 |-- block_hash: string (nullable = true)
 |-- block_number: long (nullable = true)
 |-- index: long (nullable = true)
 |-- virtual_size: long (nullable = true)
 |-- lock_time: long (nullable = true)
 |-- input_count: long (nullable = true)
 |-- output_count: long (nullable = true)
 |-- is_coinbase: boolean (nullable = true)
 |-- output_value: double (nullable = true)
 |-- outputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- index: long (nullable = true)
 |    |    |-- required_signatures: long (nullable = true)
 |    |    |-- script_asm: string (nullable = true)
 |    |    |-- script_hex: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |    |    |-- value: double (nullable = true)
 |    |    |-- address: string (nullable = true)
 |-- block_timestamp: timestamp (nullable = t

In [ ]:
(rel
    .filter(F.col("Date")>= F.current_date())
 .select(["txid", "inputs","outputs", "block_timestamp"
 ])).show()

+--------------------+--------------------+--------------------+
|                txid|              inputs|             outputs|
+--------------------+--------------------+--------------------+
|e6de4388af6aecd18...|                NULL|[{0, 1, 0 7086320...|
|e7a26b63c8e594797...|[{0, 1, , , 42949...|[{0, 1, 0 70f4763...|
|8da8565f52c9f13df...|[{0, 1, , , 42949...|[{0, 1, 0 cc166e6...|
|47c62b57293929e13...|[{0, 1, , , 42949...|[{0, 1, 0 ab3af99...|
|b193a83849acc4e9d...|[{0, 1, , , 42949...|[{0, 1, 0 15f31b0...|
|eec6359b50cf402ff...|[{0, 1, , , 42949...|[{0, 1, 0 dd5cb30...|
|a47464692505cd201...|[{0, 1, , , 42949...|[{0, 1, 0 6c688f3...|
|d4943dcc3e27b8613...|[{0, 1, , , 42949...|[{0, 1, OP_DUP OP...|
|b9b624fa3712d6090...|[{0, 1, , , 42949...|[{0, 1, 0 da40bbb...|
|05ce0e69484fd2b00...|[{0, 1, 001430786...|[{0, 1, OP_HASH16...|
|a950b8ede3be71bf0...|[{0, 1, 001469616...|[{0, 1, OP_HASH16...|
|47240f8866edc3807...|[{0, 1, , , 42949...|[{0, 1, 0 6c4709d...|
|474b5ad3300d68c74...|[{0

In [21]:
spark.sql(f"""

select * from  {Block_Table_UC}
where date = current_date
limit 10
""").show()

DataFrame[hash: string, version: bigint, mediantime: timestamp, nonce: bigint, bits: string, difficulty: double, chainwork: string, size: bigint, weight: bigint, coinbase_param: string, number: bigint, transaction_count: bigint, merkle_root: string, stripped_size: bigint, timestamp: timestamp, date: date, last_modified: timestamp]

In [11]:
(spark.sql(
    "EXPLAIN FORMATTED SELECT * FROM workspace.btc_data.blocks_source WHERE date = '2023-01-15';"
)).show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
s3_fs = pafs.S3FileSystem(anonymous=True, region="us-east-2")


In [15]:
s3 = pafs.S3FileSystem(anonymous=True, region="us-east-2")

dataset = ds.dataset(
    "aws-public-blockchain/v1.0/btc/transactions/",
    filesystem=s3,
    partitioning="hive"
)

partitions = dataset.get_fragments()

fragments = dataset.get_fragments(filter=(ds.field("date") >= "2009-01-01") & (ds.field("date") <= "2026-06-30"))
paths = sorted({f.path for f in fragments})
spark_paths = [f"s3://{p.rsplit('/', 1)[0]}/" for p in paths]
len(spark_paths)

6383

In [10]:
df = (
    spark.read
    .option("basePath", "s3://aws-public-blockchain/v1.0/btc/blocks")
    .parquet(*spark_paths)
)

In [13]:
df.collect()

[Row(hash='00000000000000000001277a81437b5d9969c72b51cede377d1f36671329345f', version=623591424, mediantime=datetime.datetime(2026, 2, 15, 12, 52, 59), nonce=3414154639, bits='17023c7e', difficulty=125864590119494.3, chainwork='00000000000000000000000000000000000000010f90de3dc1eca42bfd016576', previousblockhash='0000000000000000000110f4740107540083446a4aa5fa2b7b9d50ec6932c0dd', size=3671525, weight=3992891, coinbase_param='034c4b0e04ad1492697c204d415241204d61646520696e2055534120f09f87baf09f87b8207c763034fabe6d6db0a2cacd5d4de85554083f7c7c86320e2450a01f540ce72533aa1d4bb194c367010000000000000013fbefd9d10000000000ffffffff', number=936780, transaction_count=632, merkle_root='a9adc1db9ebe40b053853bd5fa92cfb50672fce6c3762117db5d1dec0cf89a6a', stripped_size=107122, timestamp=datetime.datetime(2026, 2, 15, 13, 48, 21), date=datetime.date(2026, 2, 15), last_modified=datetime.datetime(2026, 2, 15, 13, 47, 35, 849211)),
 Row(hash='00000000000000000001b61d223eef3a26222c91af17524d4f4388477521ef35', 

In [4]:
table_name = "workspace.btc_data.blocks"
q = f"""
    select max(number) as latest_block_number from {table_name}
    where date >= current_date - 5
    """


result = spark.sql(q).collect()[0]["latest_block_number"]

In [22]:
def get_public_block_tip(spark)-> int|None:
    """
    Returns the latest block number from the external blocks table --- public s3
    """
    s3 = pafs.S3FileSystem(anonymous=True, region="us-east-2")

    dataset = ds.dataset(
        "aws-public-blockchain/v1.0/btc/blocks/",
        filesystem=s3,
        partitioning="hive")

    partitions = dataset.get_fragments()

    fragments = dataset.get_fragments(filter=(ds.field("date") >=
                                               str(datetime.now().date() - timedelta(1))))
                    
    paths = sorted({f.path for f in fragments})
    spark_paths = [f"s3://{p.rsplit('/', 1)[0]}/" for p in paths]


    rel = (
    spark.read
    
    .option("basePath", "s3://aws-public-blockchain/v1.0/btc/blocks")
    .parquet(*spark_paths)
    .select(F.max('number')))

    return rel.collect()[0][0]


get_public_block_tip(spark)




964359

In [18]:
s3 = pafs.S3FileSystem(anonymous=True, region="us-east-2")

dataset = ds.dataset(
        "aws-public-blockchain/v1.0/btc/blocks/",
        filesystem=s3,
        partitioning="hive")

In [23]:
today = datetime.now().strftime("%Y-%m-%d")
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
spark_paths = [f"s3://aws-public-blockchain/v1.0/btc/blocks/date={d}/" for d in (yesterday, today)]

rel = (
    spark.read
    
    .option("basePath", "s3://aws-public-blockchain/v1.0/btc/blocks")
    .parquet(*spark_paths)
    .select(F.max('number')))

rel.collect()[0][0]


964360

In [6]:
TABLE_NAME = "workspace.btc_data.blocks"
S3_BLOCKS = "s3://aws-public-blockchain/v1.0/btc/blocks/"
LOOKBACK_DAYS = 3
MERGE_KEY = "number"
BLOCKS_SOURCE = "workspace.btc_data.blocks_source" # check source.py for the setup on location and table --- points to public s3
REORG_BUFFER = 10  # 6 is considered standard for reorgs to happen but set to 10 for safety


def read_blocks_source(spark,start_date:str, local_block_tip:int = 0 , reorg_buffer:int= REORG_BUFFER):

    rel = (
        spark.read.option("mergeSchema","true")
        .table(BLOCKS_SOURCE)
        .filter(f"date >= '{start_date}'")
        .filter(f"number >= {local_block_tip - reorg_buffer}")
           )

    return rel


read_blocks_source(spark,start_date = "2026-08-26",local_block_tip= 9643732 ,reorg_buffer=  REORG_BUFFER)

DataFrame[hash: string, version: bigint, mediantime: timestamp, nonce: bigint, bits: string, difficulty: double, chainwork: string, size: bigint, weight: bigint, coinbase_param: string, number: bigint, transaction_count: bigint, merkle_root: string, stripped_size: bigint, timestamp: timestamp, date: date, last_modified: timestamp, previousblockhash: string]

In [9]:

def read_blocks_source(spark,start_date:str, local_block_tip:int = 0 , reorg_buffer:int= REORG_BUFFER):

    s3 = pafs.S3FileSystem(anonymous=True, region="us-east-2")

    dataset = ds.dataset(
        "aws-public-blockchain/v1.0/btc/blocks/",
        filesystem=s3,
        partitioning="hive"
    )

    partitions = dataset.get_fragments()

    fragments = dataset.get_fragments(filter=(ds.field("date") >= start_date) )
    paths = sorted({f.path for f in fragments})
    spark_paths = [f"s3://{p.rsplit('/', 1)[0]}/" for p in paths]
    print(len(spark_paths))
    rel = (
        spark.read.option("mergeSchema","true")
        .option("basePath", "s3://aws-public-blockchain/v1.0/btc/blocks")
        .parquet(*spark_paths)
        .filter(f"date >= '{start_date}'")
        .filter(f"number >= {local_block_tip - reorg_buffer}")
           )

    return rel


read_blocks_source(spark,start_date = "2026-01-26",local_block_tip= 9643732 ,reorg_buffer=  REORG_BUFFER)

224


DataFrame[hash: string, version: bigint, mediantime: timestamp, nonce: bigint, bits: string, difficulty: double, chainwork: string, previousblockhash: string, size: bigint, weight: bigint, coinbase_param: string, number: bigint, transaction_count: bigint, merkle_root: string, stripped_size: bigint, timestamp: timestamp, date: date, last_modified: timestamp]